
In this version, we are going to recreate structure from motion by classic computer vision algorithms such as SIFT and BFMatching. Also we will use brute-force due to relatively small size of set of images. And for the record, in the next versions will be image comparison by similarity of embeddings, and usage of nerual networks for the keypoint detection and comparison. 

In [2]:
%%capture
!pip install git+https://github.com/cvg/LightGlue.git
!pip install pycolmap


In [3]:
import warnings
warnings.filterwarnings('ignore') 

In [36]:
import numpy as np

import cv2 
import os
import itertools

import tqdm
from pathlib import Path

import h5py
import plotly.graph_objects as go

import pycolmap

import sys

# Insert the folder path (exclude the file name itself)
sys.path.insert(1, '/kaggle/input/datasets/konivlarov/tgaafaf')

# Import your module normally
from database import *
from h5py_file import *

import torch
import torchvision
from torchvision.models import vgg16
from torch.nn.functional import cosine_similarity
from torchvision.transforms import v2

from lightglue import ALIKED, LightGlue
from lightglue.utils import load_image, rbd

import pickle


In [72]:
def embed_image(data_transform: v2.Compose,
                model: torchvision.models,
                path: Path | str):
    "Output: Torch.Tensor[1, 4096]"
    
    image = load_image(path)
    inputs = data_transforms(image).to(device).unsqueeze(dim = 0)
    output = model(inputs)
    
    return output




def image_similarity(
    paths: list[Path] | list[str],
    lower_border: float = 0.3,
    upper_border: float = 0.99,
    device: torch.device = 'cpu') -> list[tuple[int, int]]:
    
    data_transforms = v2.Compose([
     v2.ToDtype(torch.float64, scale=True),    
     v2.Resize((224, 224)),
     v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
     ])
    
    model = vgg16(weights='DEFAULT').to(device)
    model.classifier = model.classifier[:-1]
    
    list_of_pairs = []
    temp_i = 1
    n_images = len(paths)
    image_1 = embed_image(data_transforms, model, paths[0])

    with torch.inference_mode():

        for i in tqdm(range(n_images-1), desc="Checking image similarities"):
            if temp_i != i:
                temp_i = i
                image_1 = embed_image(data_transforms, model, paths[temp_i])
           
            for j in range(i+1, n_images):
            
                image_2 = embed_image(data_transforms, model, paths[j])
                CosSim = cosine_similarity(image_1, image_2)
                if (CosSim < upper_border) and (CosSim > lower_border): 
                    list_of_pairs.append((i, j))

    return list_of_pairs
            



def detect_keypoints(
    paths: list[Path],
    feature_dir: Path,
    max_num_keypoints: int = 4096, 
    detection_threshold: float = 0.01,
    device: torch.device = 'cpu'
    
) -> None:
    #dtype = torch.float32 # ALIKED has issues with float16
    
    extractor = ALIKED(max_num_keypoints=max_num_keypoints, 
                           detection_threshold=detection_threshold).eval().to(device)
    
    feature_dir.mkdir(parents=True, exist_ok=True)

    
    
    with h5py.File(feature_dir / "keypoints.h5", mode="w") as f_keypoints, \
         h5py.File(feature_dir / "descriptors.h5", mode="w") as f_descriptors:
        masters_dict = {}
        
        for path in tqdm(paths, desc="Computing keypoints"):
            key = path.name
            
            image = load_image(path).to(device)
            
            feats = extractor.extract(image)
            kps, ds = feats['keypoints'].squeeze(), feats['descriptors'].squeeze()

            kps, ds = kps.detach().cpu().numpy(), ds.detach().cpu().numpy()
            
            f_keypoints[key] = kps
            f_descriptors[key] = ds
            
            masters_dict[key] = feats
            
        torch.save(masters_dict, 'features.pt')
        
        
            
    
def keypoint_distances(
    paths: list[Path],
    index_pairs: list[tuple[int, int]],
    feature_dir: Path,
    min_matches: int = 15,
    verbose: bool = False,
    device: torch.device = 'cpu',
    n_layers: int = 9,
    filter_threshold: float = 0.01,
    depth_confidence: float = 0.95,
    width_confidence: float = 0.95
) -> None:

    
    matcher = LightGlue(features='aliked', 
                        n_layers = n_layers,
                        filter_threshold = filter_threshold,
                        depth_confidence = depth_confidence,
                        width_confidence = width_confidence).eval().to(device)

    features = torch.load('features.pt')
    
    with h5py.File(feature_dir / "matches.h5", mode="w") as f_matches:
            for idx1, idx2 in tqdm(index_pairs, desc="Computing keypoing distances"):
                
                key1, key2 = paths[idx1].name, paths[idx2].name

                feats0, feats1  = features[key1], features[key2]
                matches = matcher({'image0': feats0, 
                                   'image1': feats1})

                
                feats0, feats1, matches = [rbd(x) for x in 
                                             [feats0, feats1, matches]]
                matches = matches['matches'].detach().cpu().numpy()
                # We have matches to consider
                n_matches = np.shape(matches)[0]
                if n_matches:
                    if verbose:
                        print(f"{key1}-{key2}: {n_matches} matches")
                    # Store the matches in the group of one image
                    if n_matches >= min_matches:
                        group  = f_matches.require_group(key1)
                        group.create_dataset(key2, data=matches)




def import_into_colmap(
    path: Path,
    feature_dir: Path,
    database_path: str = "colmap.db",
) -> None:
    """Adds keypoints into colmap"""
    db = COLMAPDatabase.connect(database_path)
    db.create_tables()
    single_camera = False
    fname_to_id = add_keypoints(db, feature_dir, path, "", "simple-pinhole", single_camera)
    add_matches(
        db,
        feature_dir,
        fname_to_id,
    )
    db.commit()




def visualize(maps: pycolmap.Reconstruction) -> None:

  xyz = []
  rgb = []
  
  for point3D_id, point3D in maps.points3D.items():
      xyz.append(point3D.xyz)
      # PyCOLMAP stores colors as 0-255 integers; Open3D requires 0.0-1.0 floats
      rgb.append(point3D.color / 255.0)
      
  xyz_np = np.array(xyz)
  rgb_np = np.array(rgb)
  
  fig = go.Figure(data=[go.Scatter3d(
      x=xyz_np[:, 0], y=xyz_np[:, 1], z=xyz_np[:, 2],
      mode='markers',
      marker=dict(
          size=2,
          color=['rgb({},{},{})'.format(int(r*255), int(g*255), int(b*255)) for r, g, b in rgb_np],
          opacity=0.8
      )
  )])
  
  fig.update_layout(margin=dict(l=0, r=0, b=0, t=0))
  fig.show()

In [73]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

path = '/kaggle/input/competitions/image-matching-challenge-2024/test/church/images'
feature_dir = Path("./sample_test_features")
images_list = list(Path(path).glob("*.png"))

detect_keypoints(images_list, feature_dir, device = device)
all_pairs = image_similarity(images_list, device = device)
keypoint_distances(images_list, all_pairs, 
                   feature_dir, device = device,
                   n_layers = 12, depth_confidence = 0.98,
                   filter_threshold = 0.2,
                   width_confidence = 0.98)


Downloading: "https://github.com/Shiaoming/ALIKED/raw/main/models/aliked-n16.pth" to /root/.cache/torch/hub/checkpoints/aliked-n16.pth


100%|██████████| 2.61M/2.61M [00:00<00:00, 25.6MB/s]
Checking image similarities: 100%|██████████| 40/40 [00:48<00:00,  1.21s/it]


Downloading: "https://github.com/cvg/LightGlue/releases/download/v0.1_arxiv/aliked_lightglue.pth" to /root/.cache/torch/hub/checkpoints/aliked_lightglue_v0-1_arxiv.pth


100%|██████████| 45.4M/45.4M [00:00<00:00, 102MB/s] 
Computing keypoing distances: 100%|██████████| 139/139 [00:11<00:00, 12.44it/s]


In [74]:
database_path = "colmap.db"
images_dir = images_list[0].parent
import_into_colmap(
    images_dir, 
    feature_dir, 
    database_path
)

# This does RANSAC
pycolmap.match_exhaustive(database_path)

 37%|███▋      | 130/351 [00:00<00:00, 3154.43it/s]
I20260831 16:06:16.279680 138546733336128 feature_matching.cc:195] === Feature matching & geometric verification ===
I20260831 16:06:16.280359 138546602042944 sift.cc:1565] Creating SIFT CPU feature matcher
I20260831 16:06:16.280417 138546207778368 sift.cc:1565] Creating SIFT CPU feature matcher
I20260831 16:06:16.280451 138546216171072 sift.cc:1565] Creating SIFT CPU feature matcher
I20260831 16:06:16.280482 138546610435648 sift.cc:1565] Creating SIFT CPU feature matcher
I20260831 16:06:16.280768 138546733336128 pairing.cc:180] Generating exhaustive image pairs...
I20260831 16:06:16.280801 138546733336128 pairing.cc:213] Processing block [1/1, 1/1]
I20260831 16:06:16.735896 138546733336128 feature_matching.cc:217] in 0.455s
I20260831 16:06:16.735955 138546733336128 timer.cc:90] Elapsed time: 0.008 [minutes]


In [89]:
mapper_options = pycolmap.IncrementalPipelineOptions()
mapper_options.min_model_size = 2
mapper_options.max_num_models = 3
mapper_options.mapper.abs_pose_max_error = 7.0
mapper_options.mapper.filter_max_reproj_error = 6.0
mapper_options.mapper.abs_pose_min_num_inliers = 40
mapper_options.mapper.init_min_tri_angle = 6.0
mapper_options.mapper.init_min_num_inliers = 50
mapper_options.triangulation.ignore_two_view_tracks = False


maps = pycolmap.incremental_mapping(
    database_path=database_path, 
    image_path=images_dir,
    output_path=Path.cwd() / "incremental_pipeline_outputs",
    options = mapper_options
)
maps[0]

I20260831 16:15:52.602319 138554092696704 incremental_pipeline.cc:278] Loading database
I20260831 16:15:52.602445 138554092696704 database_cache.cc:72] Loading rigs...
I20260831 16:15:52.602462 138554092696704 database_cache.cc:82]  0 in 0.000s
I20260831 16:15:52.602472 138554092696704 database_cache.cc:90] Loading cameras...
I20260831 16:15:52.602580 138554092696704 database_cache.cc:108]  41 in 0.000s
I20260831 16:15:52.602593 138554092696704 database_cache.cc:116] Loading frames...
I20260831 16:15:52.602612 138554092696704 database_cache.cc:126]  0 in 0.000s
I20260831 16:15:52.602622 138554092696704 database_cache.cc:134] Loading matches...
I20260831 16:15:52.604051 138554092696704 database_cache.cc:139]  129 in 0.001s
I20260831 16:15:52.604075 138554092696704 database_cache.cc:147] Loading images...
I20260831 16:15:52.608217 138554092696704 database_cache.cc:241]  41 in 0.004s (connected 35, loaded 35)
I20260831 16:15:52.608271 138554092696704 database_cache.cc:255] Loading pose pr

Reconstruction(num_rigs=30, num_cameras=30, num_frames=30, num_reg_frames=30, num_images=30, num_points3D=13205)

In [91]:
visualize(maps[0])